In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tensorflow.keras import layers, models
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

DATA_DIR = Path("../data/casting_512x512")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

In [2]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    class_names=["ok_front", "def_front"],
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    class_names=["ok_front", "def_front"],
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    class_names=["ok_front", "def_front"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 1300 files belonging to 2 classes.
Using 1040 files for training.
Found 1300 files belonging to 2 classes.
Using 260 files for validation.
Found 1300 files belonging to 2 classes.


In [3]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
base_model.trainable = False

In [6]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [7]:
inputs = layers.Input(shape=(224, 224, 3))

x = preprocess_input(inputs)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.30)(x)

x = layers.Dense(
    64,
    activation="relu"
)(x)

x = layers.Dropout(0.20)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

mobilenet_model = models.Model(
    inputs,
    outputs
)

mobilenet_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [10]:
mobilenet_callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=0.000001
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath="../models/best_mobilenetv2_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

In [11]:
mobilenet_history = mobilenet_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=mobilenet_callbacks
)

Epoch 1/15


B:\DEEP_LEARNING\casting-quality-inspection\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


33/33 ━━━━━━━━━━━━━━━━━━━━ 26s 571ms/step - accuracy: 0.7635 - loss: 0.4950 - precision: 0.7844 - recall: 0.8301 - val_accuracy: 0.8577 - val_loss: 0.3110 - val_precision: 0.9437 - val_recall: 0.8221 - learning_rate: 0.0010
Epoch 2/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 16s 477ms/step - accuracy: 0.8712 - loss: 0.3059 - precision: 0.8993 - recall: 0.8819 - val_accuracy: 0.9115 - val_loss: 0.2233 - val_precision: 0.9321 - val_recall: 0.9264 - learning_rate: 0.0010
Epoch 3/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 469ms/step - accuracy: 0.8952 - loss: 0.2557 - precision: 0.9193 - recall: 0.9029 - val_accuracy: 0.9269 - val_loss: 0.1749 - val_precision: 0.9500 - val_recall: 0.9325 - learning_rate: 0.0010
Epoch 4/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 471ms/step - accuracy: 0.8990 - loss: 0.2376 - precision: 0.9171 - recall: 0.9126 - val_accuracy: 0.9269 - val_loss: 0.1654 - val_precision: 0.9337 - val_recall: 0.9509 - learning_rate: 0.0010
Epoch 5/15
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 464ms/step - accuracy: 0.9288

In [12]:
mobilenet_test_results = mobilenet_model.evaluate(test_dataset)

print("MobileNetV2 Test Results:")

for name, value in zip(
    mobilenet_model.metrics_names,
    mobilenet_test_results
):
    print(f"{name}: {value:.4f}")

41/41 ━━━━━━━━━━━━━━━━━━━━ 24s 580ms/step - accuracy: 0.9900 - loss: 0.0470 - precision: 0.9961 - recall: 0.9872        
MobileNetV2 Test Results:
loss: 0.0470
compile_metrics: 0.9900


In [13]:
mobilenet_probabilities = mobilenet_model.predict(
    test_dataset
).flatten()

mobilenet_predictions = (
    mobilenet_probabilities >= 0.5
).astype(int)

41/41 ━━━━━━━━━━━━━━━━━━━━ 35s 835ms/step


In [14]:
mobilenet_actual = np.concatenate([
    labels.numpy().flatten()
    for images, labels in test_dataset
]).astype(int)

In [15]:
print(
    classification_report(
        mobilenet_actual,
        mobilenet_predictions,
        target_names=[
            "Non-defective",
            "Defective"
        ]
    )
)

               precision    recall  f1-score   support

Non-defective       0.98      0.99      0.99       519
    Defective       1.00      0.99      0.99       781

     accuracy                           0.99      1300
    macro avg       0.99      0.99      0.99      1300
 weighted avg       0.99      0.99      0.99      1300



In [16]:
mobilenet_matrix = confusion_matrix(
    mobilenet_actual,
    mobilenet_predictions
)

print(mobilenet_matrix)

[[516   3]
 [ 10 771]]


In [17]:
tn, fp, fn, tp = mobilenet_matrix.ravel()

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)

True Negatives : 516
False Positives: 3
False Negatives: 10
True Positives  : 771


In [18]:
mobilenet_accuracy = accuracy_score(
    mobilenet_actual,
    mobilenet_predictions
)

mobilenet_precision = precision_score(
    mobilenet_actual,
    mobilenet_predictions
)

mobilenet_recall = recall_score(
    mobilenet_actual,
    mobilenet_predictions
)

mobilenet_f1 = f1_score(
    mobilenet_actual,
    mobilenet_predictions
)

print(f"Accuracy : {mobilenet_accuracy:.4f}")
print(f"Precision: {mobilenet_precision:.4f}")
print(f"Recall   : {mobilenet_recall:.4f}")
print(f"F1       : {mobilenet_f1:.4f}")

Accuracy : 0.9900
Precision: 0.9961
Recall   : 0.9872
F1       : 0.9916
